BASIC INSTALL

In [ ]:
!pip -q install pyserini faiss-cpu sentence-transformers huggingface_hub
!apt-get -qq install -y openjdk-21-jdk-headless > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


CONFIG AND DOWNLOADING SHARDS

In [ ]:
import gzip, json, os
from huggingface_hub import hf_hub_download

NUM_SHARDS = 2          # 2 shards ~ 0.5M passages. Full English corpus = 66
WORK = "/content/miracl_en"
os.makedirs(WORK, exist_ok=True)

shard_paths = []
for i in range(NUM_SHARDS):
    p = hf_hub_download(
        repo_id="miracl/miracl-corpus",
        repo_type="dataset",
        filename=f"miracl-corpus-v1.0-en/docs-{i}.jsonl.gz",
    )
    shard_paths.append(p)

def iter_passages():
    for p in shard_paths:
        with gzip.open(p, "rt", encoding="utf-8") as f:
            for line in f:
                d = json.loads(line)          # keys: docid, title, text
                yield d["docid"], d["title"], d["text"]

print(sum(1 for _ in iter_passages()), "passages")

miracl-corpus-v1.0-en/docs-0.jsonl.gz: reconstructing file:   0%|          |  0.00B / 98.9MB            

miracl-corpus-v1.0-en/docs-0.jsonl.gz: downloading bytes:           |  0.00B            

miracl-corpus-v1.0-en/docs-1.jsonl.gz: reconstructing file:   0%|          |  0.00B / 97.3MB            

miracl-corpus-v1.0-en/docs-1.jsonl.gz: downloading bytes:           |  0.00B            

1000000 passages


Index 1, BM25 (Pyserini / Lucene)

In [ ]:
# Convert to Pyserini's JsonCollection format
bm25_in = f"{WORK}/bm25_input"
os.makedirs(bm25_in, exist_ok=True)
with open(f"{bm25_in}/docs.jsonl", "w", encoding="utf-8") as out:
    for docid, title, text in iter_passages():
        out.write(json.dumps({"id": docid, "contents": f"{title} {text}"}) + "\n")

!python -m pyserini.index.lucene \
  --collection JsonCollection \
  --input {bm25_in} \
  --index {WORK}/bm25_index \
  --generator DefaultLuceneDocumentGenerator \
  --threads 4 --storePositions --storeDocvectors --storeRaw

[0.014s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.014s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-10-24 18:27:39,456 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:208) - Setting log level to INFO
2026-10-24 18:27:39,462 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:211) - ============ Loading Index Configuration ============
2026-10-24 18:27:39,462 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:212) - AbstractIndexer settings:
2026-10-24 18:27:39,465 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:213) -  + DocumentCollection path: /content/miracl_en/bm25_input
2026-10-24 18:27:39,465 INFO  [main] index.AbstractIndexer (AbstractIndexer.java:214) -  + CollectionClass: JsonCollection
2026-10-24 18:27:39,465 INFO  [m

TEST IT

In [ ]:
from pyserini.search.lucene import LuceneSearcher
import json

bm25 = LuceneSearcher(f"{WORK}/bm25_index") #INDEX
bm25.set_bm25(k1=0.9, b=0.4)     # Pyserini defaults for MS MARCO-style passages

def bm25_search(q, k=5):
    return [(h.docid, h.score, json.loads(h.lucene_document.get("raw"))["contents"][:200])
            for h in bm25.search(q, k)]

for r in bm25_search("who discovered penicillin"):
    print(r)

('1937#15', 10.348899841308594, 'Alexander Fleming Fleming also discovered very early that bacteria developed antibiotic resistance whenever too little penicillin was used or when it was used for too short a period. Almroth Wright ha')
('1937#13', 10.200900077819824, 'Alexander Fleming Fleming was modest about his part in the development of penicillin, describing his fame as the "Fleming Myth" and he praised Florey and Chain for transforming the laboratory curiosit')
('23312#0', 9.868000030517578, 'Penicillin Penicillin (PCN or pen) is a group of antibiotics which include penicillin G (intravenous use), penicillin V (use by mouth), procaine penicillin, and benzathine penicillin (intramuscular us')
('1805#42', 9.345100402832031, 'Antibiotic Ernst Chain, Howard Florey and Edward Abraham succeeded in purifying the first penicillin, penicillin G, in 1942, but it did not become widely available outside the Allied military before 1')
('23312#18', 8.91469955444336, 'Penicillin Despite the lac

Index 2, Dense (multilingual-e5 + FAISS)

In [ ]:
import numpy as np, faiss, torch
from sentence_transformers import SentenceTransformer

MODEL = "intfloat/multilingual-e5-small"     # 384-dim. Try multilingual-e5-base for better quality
model = SentenceTransformer(MODEL, device="cuda")
model.half()
model.max_seq_length = 256
DIM = model.get_sentence_embedding_dimension()

# Count passages, then encode into a disk-backed array
N = sum(1 for _ in iter_passages())
emb = np.lib.format.open_memmap(f"{WORK}/emb.npy", mode="w+", dtype="float16", shape=(N, DIM))
docids = []

BATCH, CHUNK = 128, 20_000
buf_txt, buf_ids, pos = [], [], 0

def flush():
    global pos, buf_txt, buf_ids
    if not buf_txt: return
    v = model.encode(buf_txt, batch_size=BATCH, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=False)
    emb[pos:pos+len(v)] = v
    docids.extend(buf_ids)
    pos += len(v)
    print(f"{pos}/{N}", end="\r")
    buf_txt, buf_ids = [], []

for docid, title, text in iter_passages():
    buf_txt.append(f"passage: {title} {text}")
    buf_ids.append(docid)
    if len(buf_txt) == CHUNK: flush()
flush()
emb.flush()

json.dump(docids, open(f"{WORK}/docids.json", "w"))

# Build FAISS index (exact inner product = cosine on normalized vectors)
index = faiss.IndexFlatIP(DIM)
for s in range(0, N, 100_000):
    index.add(np.asarray(emb[s:s+100_000], dtype="float32"))
faiss.write_index(index, f"{WORK}/dense.faiss")
print("\nDense index size:", index.ntotal)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_2471/638618089.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DIM = model.get_sentence_embedding_dimension()



Dense index size: 1000000


TEST IT

In [ ]:
docid2text = {}   # lazy lookup; for a small corpus we can cache text
for docid, title, text in iter_passages():
    docid2text[docid] = f"{title} {text}"

def dense_search(q, k=5):
    qv = model.encode([f"query: {q}"], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idx = index.search(qv, k)
    return [(docids[i], float(s), docid2text[docids[i]][:200]) for s, i in zip(scores[0], idx[0])]

for r in dense_search("who discovered penicillin"):
    print(r)

('37220#63', 0.9161660075187683, "Infection Alexander Fleming discovered the world's first antibiotic, Penicillin, which Florey and Chain then developed.")
('23312#26', 0.9081426858901978, 'Penicillin The chemical structure of penicillin was first proposed by Edward Abraham in 1942 and was later confirmed in 1945 using X-ray crystallography by Dorothy Crowfoot Hodgkin, who was also worki')
('23312#0', 0.9003922939300537, 'Penicillin Penicillin (PCN or pen) is a group of antibiotics which include penicillin G (intravenous use), penicillin V (use by mouth), procaine penicillin, and benzathine penicillin (intramuscular us')
('1937#17', 0.8956326246261597, "Alexander Fleming Fleming's discovery of penicillin changed the world of modern medicine by introducing the age of useful antibiotics; penicillin has saved, and is still saving, millions of people aro")
('23312#16', 0.8913052082061768, 'Penicillin Once Fleming made his discovery he grew a pure culture and discovered it was a "Penicilliu

In [ ]:
import os, json, shutil
from google.colab import drive

drive.mount("/content/drive")

WORK = "/content/miracl_en"
DEST = "/content/drive/MyDrive/miracl_en_indexes"
os.makedirs(DEST, exist_ok=True)

# 1. Passages (so dense search works standalone, no corpus shards needed later)
with open(f"{DEST}/passages.jsonl", "w", encoding="utf-8") as f:
    for docid, title, text in iter_passages():
        f.write(json.dumps({"id": docid, "title": title, "text": text}) + "\n")

# 2. BM25 index (folder)
shutil.copytree(f"{WORK}/bm25_index", f"{DEST}/bm25_index", dirs_exist_ok=True)

# 3. Dense index + id mapping
shutil.copy(f"{WORK}/dense.faiss", DEST)
shutil.copy(f"{WORK}/docids.json", DEST)

# Check
for root, _, files in os.walk(DEST):
    for fn in files:
        p = os.path.join(root, fn)
        print(f"{os.path.getsize(p)/1e6:10.1f} MB  {os.path.relpath(p, DEST)}")
print("\nDone. Folder:", DEST)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     575.3 MB  passages.jsonl
    1536.0 MB  dense.faiss
      11.6 MB  docids.json
       9.2 MB  bm25_index/_0_Lucene90_0.dvd
       0.0 MB  bm25_index/_0_Lucene104_0.tmd
     276.0 MB  bm25_index/_0.tvd
       0.0 MB  bm25_index/segments_1
       0.0 MB  bm25_index/_0.fnm
      10.3 MB  bm25_index/_0_Lucene104_0.tim
       0.0 MB  bm25_index/_0.tvm
     359.3 MB  bm25_index/_0.fdt
      76.4 MB  bm25_index/_0_Lucene104_0.doc
       0.0 MB  bm25_index/_0_Lucene90_0.dvm
       0.0 MB  bm25_index/_0_Lucene104_0.psm
       0.0 MB  bm25_index/_0_Lucene90_0.dvs
       0.3 MB  bm25_index/_0_Lucene104_0.tip
       0.2 MB  bm25_index/_0.tvx
      59.2 MB  bm25_index/_0_Lucene104_0.pos
       0.0 MB  bm25_index/_0.fdm
       0.0 MB  bm25_index/_0.si
       1.0 MB  bm25_index/_0.nvd
       0.0 MB  bm25_index/_0.fdx
       0.0 MB  bm25_index/_0.nvm
       0.0 MB  bm25